### ATLAS INSURANCE DATA WAREHOUSE LOADER 
---
**Purpose:** Load dimension and fact tables from Silver layer to Gold (Warehouse) layer with proper schema organization and naming conventions.

**Author:** Data Engineering Team 

**Last update date:** 2026-01-20

In [22]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, DoubleType, DateType
import com.microsoft.spark.fabric
from pyspark.sql.functions import *
import pyspark.sql.functions as F
from pyspark.sql import Row, Window
from datetime import datetime, timedelta
import calendar

StatementMeta(, 6966b8ce-7ba6-4699-9088-801789e4c282, 24, Finished, Available, Finished)

###### **Data Extraction:** Load tables from Silver layer

In [23]:
agents_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.agents")
applicants_addresses_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_addresses")
applicants_banking_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_banking")
applicants_contacts_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_contacts")
applicants_employment_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_employment")
applicants_health_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.applicants_health")
claim_processing_details_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.claim_processng_details")
claims_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.claims")
demographic_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.demographic")
fact_applicants_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.fact_applicants")
insurance_policies_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.insurance_policies")
nationality_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.nationality")
payment_history_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.payment_history")
policies_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.policies")
policies_dates_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.policies_dates")
policy_coverages_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.policy_coverages")
reinsurance_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.reinsurance")
reinsurance_companies_df = spark.table("lk_atlas_insurance_data_SILVER.dbo.reinsurance_companies")

StatementMeta(, 6966b8ce-7ba6-4699-9088-801789e4c282, 25, Finished, Available, Finished)

###### **Schema design**: Business domain organization

In [24]:
# Schema layout follows business domains: applicants, policies, claims, premiums
# Tables are prefixed with dim_ for dimensions and fact_ for facts
tables = {
    "applicants": {
        "dim": [
            "dim_addresses",          # Applicant address information
            "dim_banking",            # Banking details for premium payments
            "dim_contacts",           # Contact information
            "dim_employment",         # Employment history
            "dim_health",             # Health and medical information
            "dim_demographic",        # Demographic attributes
            "dim_nationality"         # Nationality and citizenship
        ],
        "fact": [
            "fact_applicants"         # Core applicant facts and metrics
        ]
    },
    "policies": {
        "dim": [
            "dim_agents",                 # Insurance agents/sales representatives
            "dim_policies_dates",         # Policy lifecycle dates
            "dim_reinsurance_companies"   # Reinsurance partners
        ],
        "fact": [
            "fact_policies",              # Policy master records
            "fact_policy_coverages",      # Coverage details and limits
            "fact_insurance_policies",    # Policy financial terms
            "fact_reinsurance"            # Risk transfer arrangements
        ]
    },
    "claims": {
        "fact": [
            "fact_claims"                 # Claim transactions and amounts
        ], 
        "dim": [
            "dim_claim_processing_details" # Claim adjudication workflow
        ]
    },
    "premiums": {
        "fact": [
            "fact_payment_history"        # Premium payment transactions
        ]
    }
}

# -------------------------------------------------------------------------
# TABLE MAPPING: Transformation from Silver to Gold layer
# -------------------------------------------------------------------------

# Maps original Silver table names to (target_schema, table_type, new_table_name)
# This decouples source naming from warehouse naming conventions

table_mapping = {
    # Applicants domain transformations
    "applicants_addresses": ("applicants", "dim", "dim_addresses"),
    "applicants_banking": ("applicants", "dim", "dim_banking"),
    "applicants_contacts": ("applicants", "dim", "dim_contacts"),
    "applicants_employment": ("applicants", "dim", "dim_employment"),
    "applicants_health": ("applicants", "dim", "dim_health"),
    "demographic": ("applicants", "dim", "dim_demographic"),
    "nationality": ("applicants", "dim", "dim_nationality"),
    "fact_applicants": ("applicants", "fact", "fact_applicants"),
    
    # Policies domain transformations
    "agents": ("policies", "dim", "dim_agents"),
    "policies_dates": ("policies", "dim", "dim_policies_dates"),
    "reinsurance_companies": ("policies", "dim", "dim_reinsurance_companies"),
    "policies": ("policies", "fact", "fact_policies"),
    "policy_coverages": ("policies", "fact", "fact_policy_coverages"),
    "insurance_policies": ("policies", "fact", "fact_insurance_policies"),
    "reinsurance": ("policies", "fact", "fact_reinsurance"),
    
    # Claims domain transformations
    "claims": ("claims", "fact", "fact_claims"),
    "claim_processing_details": ("claims", "dim", "dim_claim_processing_details"),
    
    # Premiums domain transformations
    "payment_history": ("premiums", "fact", "fact_payment_history")
}

# -------------------------------------------------------------------------
# DATA CATALOG: Central registry of DataFrames
# -------------------------------------------------------------------------

# Dictionary for easy DataFrame access and management
table_dataframes = {
    "agents": agents_df,
    "applicants_addresses": applicants_addresses_df,
    "applicants_banking": applicants_banking_df,
    "applicants_contacts": applicants_contacts_df,
    "applicants_employment": applicants_employment_df,
    "applicants_health": applicants_health_df,
    "claim_processing_details": claim_processing_details_df,
    "claims": claims_df,
    "demographic": demographic_df,
    "fact_applicants": fact_applicants_df,
    "insurance_policies": insurance_policies_df,
    "nationality": nationality_df,
    "payment_history": payment_history_df,
    "policies": policies_df,
    "policies_dates": policies_dates_df,
    "policy_coverages": policy_coverages_df,
    "reinsurance": reinsurance_df,
    "reinsurance_companies": reinsurance_companies_df
}


StatementMeta(, 6966b8ce-7ba6-4699-9088-801789e4c282, 26, Finished, Available, Finished)

###### **Create Date Table**

In [25]:
# -------------------------
# CONFIGURATION & SETUP
# -------------------------

spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# -------------------------
# GET MIN/MAX DATES FROM SOURCE DATA
# -------------------------

# Find min date from fact_applicants date_of_birth (handle nulls)
min_date_df = policies_dates_df.select(
    F.min(F.coalesce(F.col("effective_date"), F.lit("2100-01-01"))).alias("min_date")
)

# Find max date from payment_history payment_date (handle nulls)
max_date_df = payment_history_df.select(
    F.max(F.coalesce(F.col("payment_date"), F.lit("1900-01-01"))).alias("max_date")
)

# Collect the dates
min_date_result = min_date_df.collect()[0]["min_date"]
max_date_result = max_date_df.collect()[0]["max_date"]

# Convert to date objects if they're not already
if isinstance(min_date_result, datetime):
    min_date = min_date_result.date()
elif isinstance(min_date_result, str):
    min_date = datetime.strptime(min_date_result, "%Y-%m-%d").date()
else:
    min_date = min_date_result

if isinstance(max_date_result, datetime):
    max_date = max_date_result.date()
elif isinstance(max_date_result, str):
    max_date = datetime.strptime(max_date_result, "%Y-%m-%d").date()
else:
    max_date = max_date_result

# Add buffer for future dates (1 year buffer)
max_date_with_buffer = max_date + timedelta(days=365)

print(f"Min Date: {min_date}")
print(f"Max Date: {max_date}")
print(f"Max Date with Buffer: {max_date_with_buffer}")

# -------------------------
# DATE DIMENSION GENERATION FUNCTION (WITH FIXED SCHEMA)
# -------------------------

def generate_date_dimension(start_date, end_date):
    """
    Generate a date dimension table between start_date and end_date
    """
    # Create a list of all dates in the range
    date_range = []
    current_date = start_date
    
    while current_date <= end_date:
        date_range.append(current_date)
        current_date += timedelta(days=1)
    
    # Create DataFrame from date range
    dates_df = spark.createDataFrame([(d,) for d in date_range], ["Date"])
    
    # Ensure Date column is DateType
    dates_df = dates_df.withColumn("Date", F.col("Date").cast(DateType()))
    
    # Year, Quarter, Month
    dates_df = dates_df.withColumn("Year", F.year("Date"))
    dates_df = dates_df.withColumn("Quarter", F.quarter("Date"))
    dates_df = dates_df.withColumn("QuarterName", 
        F.when(F.col("Quarter") == 1, "Q1")
         .when(F.col("Quarter") == 2, "Q2")
         .when(F.col("Quarter") == 3, "Q3")
         .when(F.col("Quarter") == 4, "Q4")
         .otherwise("Unknown"))
    dates_df = dates_df.withColumn("Month", F.month("Date"))
    dates_df = dates_df.withColumn("MonthName", 
        F.date_format("Date", "MMMM"))
    
    # Week number (ISO week)
    dates_df = dates_df.withColumn("WeekNumber", F.weekofyear("Date"))
    
    # Day
    dates_df = dates_df.withColumn("Day", F.dayofmonth("Date"))
    
    # Fix for DayOfWeek - using dayofweek function (Sunday=1, Saturday=7)
    # Then convert to Monday=1, Sunday=7 format
    dates_df = dates_df.withColumn("DayOfWeek",
        F.when(F.dayofweek("Date") == 1, 7)  # Sunday becomes 7
         .otherwise(F.dayofweek("Date") - 1))  # Monday becomes 1, etc.
    
    dates_df = dates_df.withColumn("DayName", 
        F.date_format("Date", "EEEE"))
    dates_df = dates_df.withColumn("DayOfYear", F.dayofyear("Date"))
    
    # Year-Month (YYYY-MM format)
    dates_df = dates_df.withColumn("YearMonth", 
        F.date_format("Date", "yyyy-MM"))
    
    # Month-Year (MM-YYYY format)
    dates_df = dates_df.withColumn("MonthYear", 
        F.date_format("Date", "MM-yyyy"))
    
    # IsWeekend - cast to BooleanType to match existing schema
    dates_df = dates_df.withColumn("IsWeekend", 
        F.when((F.col("DayOfWeek") == 6) | (F.col("DayOfWeek") == 7), True)
         .otherwise(False).cast("boolean"))
    
    # IsWorkingDay - cast to BooleanType to match existing schema
    dates_df = dates_df.withColumn("IsWorkingDay", 
        F.when(F.col("IsWeekend") == True, False)
         .otherwise(True).cast("boolean"))
    
    # IsLastDayOfMonth - cast to BooleanType to match existing schema
    dates_df = dates_df.withColumn("IsLastDayOfMonth",
        F.when(F.col("Day") == F.dayofmonth(F.last_day("Date")), True)
         .otherwise(False).cast("boolean"))
    
    # Fiscal Year (assuming fiscal year starts April 1)
    dates_df = dates_df.withColumn("FiscalYear",
        F.when(F.col("Month") >= 4, F.col("Year"))
         .otherwise(F.col("Year") - 1))
    
    # Fiscal Quarter (adjust for April start)
    dates_df = dates_df.withColumn("FiscalQuarter",
        F.when(F.col("Month") >= 4, 
            F.when(F.col("Month") <= 6, 1)
             .when(F.col("Month") <= 9, 2)
             .when(F.col("Month") <= 12, 3)
             .otherwise(4))
         .otherwise(4))  # Jan-Mar = Q4 of previous fiscal year
    
    # Add date key (YYYYMMDD format)
    dates_df = dates_df.withColumn("DateKey", 
        F.date_format("Date", "yyyyMMdd").cast(IntegerType()))
    
    # Add a timestamp for when this record was created
    dates_df = dates_df.withColumn("CreatedDate", F.current_timestamp())
    
    # Add LastUpdated timestamp
    dates_df = dates_df.withColumn("LastUpdated", F.current_timestamp())
    
    return dates_df

# -------------------------
# CHECK EXISTING TABLE & DETERMINE UPDATE NEEDED
# -------------------------

# Check if target table exists
table_exists = spark.catalog.tableExists("wh_atlas_insurance_data.dbo.dim_date_table")

if table_exists:
    print("Table exists, checking for updates...")
    
    # Get existing max date
    existing_max_date_df = spark.table("wh_atlas_insurance_data.dbo.dim_date_table") \
        .select(F.max("Date").alias("max_date"))
    
    existing_max_date = existing_max_date_df.collect()[0]["max_date"]
    
    if existing_max_date is None:
        print("Table exists but is empty")
        # Generate full date range
        start_date = min_date
        end_date = max_date_with_buffer
        needs_update = True
    else:
        print(f"Existing max date in table: {existing_max_date}")
        
        # If our new max date is greater than existing, we need to add new dates
        if max_date_with_buffer > existing_max_date:
            print(f"New dates needed from {existing_max_date} to {max_date_with_buffer}")
            start_date = existing_max_date + timedelta(days=1)
            end_date = max_date_with_buffer
            needs_update = True
        else:
            print("No new dates needed")
            needs_update = False
else:
    print("Table doesn't exist, creating from scratch...")
    # Generate full date range
    start_date = min_date
    end_date = max_date_with_buffer
    needs_update = True
    existing_max_date = None

# -------------------------
# GENERATE DATE DIMENSION DATA
# -------------------------

if needs_update:
    print(f"Generating date dimension from {start_date} to {end_date}")
    
    # Generate the new date dimension segment
    new_dates_df = generate_date_dimension(start_date, end_date)
    
    record_count = new_dates_df.count()
    print(f"Generated {record_count} new date records")
    
    # Show schema and sample
    print("\nSchema (with corrected boolean types):")
    new_dates_df.printSchema()
    
    # Show sample
    print("\nSample of new dates (first 10 rows):")
    sample_df = new_dates_df.select("Date", "Year", "MonthName", "DayName", "IsWeekend", "IsWorkingDay").limit(10)
    display(sample_df)
    
    # -------------------------
    # WRITE DATA TO TARGET TABLE
    # -------------------------
    
    if table_exists and existing_max_date is not None:
        # Append to existing table
        print("\nAppending to existing date dimension table...")
        
        # Check for duplicates before appending
        existing_dates = spark.table("wh_atlas_insurance_data.dbo.dim_date_table").select("Date")
        new_dates_filtered = new_dates_df.join(existing_dates, ["Date"], "left_anti")
        
        filtered_count = new_dates_filtered.count()
        
        if filtered_count > 0:
            print(f"Appending {filtered_count} new records (after removing duplicates)")
            
            # Append new records
            new_dates_filtered.write \
                .mode("append") \
                .saveAsTable("wh_atlas_insurance_data.dbo.dim_date_table")
            
        else:
            print("No new records to append (all dates already exist)")
    else:
        # Create new table
        print("\nCreating new date dimension table...")
        
        print(f"Creating table with {record_count} records from {start_date} to {end_date}")
        
        # OPTION 1: Use overwrite mode with option to overwrite schema
        try:
            new_dates_df.write \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .synapsesql("wh_atlas_insurance_data.dbo.dim_date_table")
            print("Date dimension table created successfully using overwriteSchema")
        except Exception as e:
            print(f"Overwrite with schema option failed: {e}")
            
            # OPTION 2: Use SQL to create table (more reliable)
            print("\nTrying alternative approach using SQL CREATE OR REPLACE...")
            
            # Create temp view
            new_dates_df.createOrReplaceTempView("temp_dim_date")
            
            # Use SQL to create/replace table
            spark.sql(f"""
                CREATE OR REPLACE TABLE wh_atlas_insurance_data.dbo.dim_date_table
                USING DELTA
                AS SELECT * FROM temp_dim_date
            """)
            
            print("Date dimension table created successfully using SQL CREATE OR REPLACE")

# -------------------------
# CLEANUP & FINALIZATION
# -------------------------

# Reset the legacy parser if you want to return to default
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

print("\nDate dimension processing complete!")
print("=" * 50)

StatementMeta(, 6966b8ce-7ba6-4699-9088-801789e4c282, 27, Finished, Available, Finished)

Min Date: 2015-01-01
Max Date: 2022-01-29
Max Date with Buffer: 2023-01-29
Table exists, checking for updates...
Existing max date in table: 2023-01-29
No new dates needed

Date dimension processing complete!


###### **Security:**  PII and sensitive data handling

In [26]:
def remove_sensitive_columns(df):
    """
    Remove personally identifiable information (PII) and sensitive data
    before loading to warehouse layer for compliance with data privacy regulations.
    """
    sensitive_columns = ["id_number", "full_name", "age", "bank_account_number", "contact"]
    
    # Only remove columns that actually exist in the DataFrame
    existing_columns = [col for col in sensitive_columns if col in df.columns]
    
    for col in existing_columns:
        df = df.drop(col)
    
    return df

StatementMeta(, 6966b8ce-7ba6-4699-9088-801789e4c282, 28, Finished, Available, Finished)

###### **Data Loading:** Core warehosue load function

In [27]:
def save_to_warehouse(df, original_name, database="wh_atlas_insurance_data"):
    # Validate table exists in mapping configuration
    if original_name not in table_mapping:
        print(f"Configuration error: {original_name} not found in table_mapping")
        return False
    
    # Extract target location from mapping
    schema, table_type, new_name = table_mapping[original_name]
    full_table_name = f"{database}.{schema}.{new_name}"
    
    # Initialize schema comparison variables
    extra_cols = set()
    missing_cols = set()
    
    try:
        print(f"Loading {original_name} → {new_name} to {schema} schema...")
        
        # Apply PII removal for compliance
        secure_df = remove_sensitive_columns(df)
        
        # Check if table exists and get target schema
        table_exists = spark.catalog.tableExists(full_table_name)
        
        if table_exists:
            print("Table exists, checking schema compatibility...")
            target_columns = set(spark.table(full_table_name).columns)
            source_columns = set(secure_df.columns)
            
            # Identify schema differences
            extra_cols = source_columns - target_columns - {"load_timestamp"}
            missing_cols = target_columns - source_columns - {"load_timestamp"}
            
            if extra_cols:
                print(f"  Extra in df: {', '.join(extra_cols)}")
                # Remove extra columns that don't exist in target
                secure_df = secure_df.drop(*extra_cols)
            
            if missing_cols:
                print(f"  Missing in df: {', '.join(missing_cols)}")
                # Add missing columns with null values
                from pyspark.sql.functions import lit
                from pyspark.sql.types import StringType
                
                # Get target table schema to determine proper types
                target_schema = spark.table(full_table_name).schema
                for col_name in missing_cols:
                    # Find the column type from target schema
                    target_col = next((field for field in target_schema if field.name == col_name), None)
                    if target_col:
                        secure_df = secure_df.withColumn(col_name, lit(None).cast(target_col.dataType))
                    else:
                        # Default to string if type not found
                        secure_df = secure_df.withColumn(col_name, lit(None).cast(StringType()))
            
            if not extra_cols and not missing_cols:
                print("Schemas match, using synapsesql...")
        
        # Add audit timestamp for data lineage
        timestamped_df = secure_df.withColumn("load_timestamp", current_timestamp())
        
        # Execute warehouse load
        if table_exists and (extra_cols or missing_cols):
            print("Schema mismatch detected. Attempting to fix...")
            # Try with overwriteSchema when there are differences
            timestamped_df.write \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .saveAsTable(full_table_name)
        else:
            # Use synapsesql for schema-compatible loads or new tables
            timestamped_df.write \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .synapsesql(full_table_name)
        
        # FIXED: Now extra_cols and missing_cols are always defined
        load_method = 'synapsesql' if not (extra_cols or missing_cols) else 'saveAsTable'
        print(f"Successfully loaded to {full_table_name} using {load_method}")
        return True
        
    except Exception as e:
        print(f"Load failed: {str(e)}")
        return False

StatementMeta(, 6966b8ce-7ba6-4699-9088-801789e4c282, 29, Finished, Available, Finished)

###### **Execution:** Main load orchestration

In [28]:
# Performance tracking across business domains
load_stats = {}

# Process each business domain sequentially
for schema_name, schema_tables in tables.items():  # Changed: schema_tables is dict with "dim" and "fact" keys
    print(f"\n{'=' * 60}")
    print(f"PROCESSING {schema_name.upper()} DOMAIN")
    print('=' * 60)
    
    success_count = 0
    fail_count = 0
    
    # Load dimension tables in this domain
    if "dim" in schema_tables:
        for dim_table_name in schema_tables["dim"]:
            
            # First, find which original table maps to this dim table
            original_name = None
            for orig_name, (sch, typ, new_name) in table_mapping.items():
                if sch == schema_name and typ == "dim" and new_name == dim_table_name:
                    original_name = orig_name
                    break
            
            if original_name and original_name in table_dataframes:
                success = save_to_warehouse(
                    table_dataframes[original_name], 
                    original_name
                )
                if success:
                    success_count += 1
                else:
                    fail_count += 1
            else:
                print(f"Cannot find source for dimension table: {dim_table_name}")
                fail_count += 1
    
    # Load fact tables in this domain
    if "fact" in schema_tables:
        for fact_table_name in schema_tables["fact"]:
            # Find the original table name
            original_name = None
            for orig_name, (sch, typ, new_name) in table_mapping.items():
                if sch == schema_name and typ == "fact" and new_name == fact_table_name:
                    original_name = orig_name
                    break
            
            if original_name and original_name in table_dataframes:
                success = save_to_warehouse(
                    table_dataframes[original_name], 
                    original_name
                )
                if success:
                    success_count += 1
                else:
                    fail_count += 1
            else:
                print(f"Cannot find source for fact table: {fact_table_name}")
                fail_count += 1
    
    # Record domain-level statistics
    load_stats[schema_name] = {"success": success_count, "fail": fail_count}
    print(f"Domain summary: {success_count} succeeded, {fail_count} failed")

StatementMeta(, 6966b8ce-7ba6-4699-9088-801789e4c282, 30, Finished, Available, Finished)


PROCESSING APPLICANTS DOMAIN
Loading applicants_addresses → dim_addresses to applicants schema...
Table exists, checking schema compatibility...
Schemas match, using synapsesql...
Successfully loaded to wh_atlas_insurance_data.applicants.dim_addresses using synapsesql
Loading applicants_banking → dim_banking to applicants schema...
Table exists, checking schema compatibility...
Schemas match, using synapsesql...
Successfully loaded to wh_atlas_insurance_data.applicants.dim_banking using synapsesql
Loading applicants_contacts → dim_contacts to applicants schema...
Table exists, checking schema compatibility...
Schemas match, using synapsesql...
Successfully loaded to wh_atlas_insurance_data.applicants.dim_contacts using synapsesql
Loading applicants_employment → dim_employment to applicants schema...
Table exists, checking schema compatibility...
Schemas match, using synapsesql...
Successfully loaded to wh_atlas_insurance_data.applicants.dim_employment using synapsesql
Loading applican

###### **Reporting:** Load completion summary

In [29]:
print(f"\n{'=' * 60}")
print("WAREHOUSE LOAD COMPLETION REPORT")
print('=' * 60)

total_success = 0
total_fail = 0

# Generate domain-level performance report
for schema_name, stats in load_stats.items():
    total_success += stats["success"]
    total_fail += stats["fail"]
    print(f"{schema_name.upper():12} : {stats['success']:2} succeeded, {stats['fail']:2} failed")

# Overall success metrics
print(f"\n{'─' * 40}")
print(f"TOTAL TABLES   : {total_success + total_fail:2}")
print(f"SUCCESSFUL     : {total_success:2} ({total_success/(total_success+total_fail)*100:.1f}%)")
print(f"FAILED         : {total_fail:2}")
print(f"{'─' * 40}")

if total_fail == 0:
    print("All tables loaded successfully to warehouse")
else:
    print(f"{total_fail} tables failed to load - review logs for details")

print(f"\nLoad process completed at: {datetime.now()}")

StatementMeta(, 6966b8ce-7ba6-4699-9088-801789e4c282, 31, Finished, Available, Finished)


WAREHOUSE LOAD COMPLETION REPORT
APPLICANTS   :  8 succeeded,  0 failed
POLICIES     :  7 succeeded,  0 failed
CLAIMS       :  2 succeeded,  0 failed
PREMIUMS     :  1 succeeded,  0 failed

────────────────────────────────────────
TOTAL TABLES   : 18
SUCCESSFUL     : 18 (100.0%)
FAILED         :  0
────────────────────────────────────────
All tables loaded successfully to warehouse

Load process completed at: 2026-01-28 20:46:01.954895
